# Sea Ice Trend Analysis (Climate Year)

Loads the climate-year (March 15 – March 14) re-indexed concentration/thickness
dataset exported by `seaice_daily_heatmaps.ipynb` and computes spring-melt /
fall-freeze trends for both fields.

Because every season is fully contained in a single row (DOY 0 = March 15,
DOY 364 = March 14 of the following year), there is no Dec/Jan year-boundary
wraparound to handle -- unlike the calendar-year version of this analysis.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import climate_year_trend_utils as cyt

## User configuration

In [ ]:
# ===== USER CONFIGURATION =====

# --- Input dataset (produced by the export cell in seaice_daily_heatmaps.ipynb) ---
route_label = 'text_route'   # must match the route_label used when the .nc was saved
nc_path     = os.path.join('.', f'seaice_climate_year_route{route_label}.nc')

# --- Output ---
save_figs  = True
output_dir = os.path.join('.', 'figures')

# --- Ensemble statistic to analyse ---
# Options: 'median', '5th_percentile', '95th_percentile'
trend_stat = '95th_percentile'

# --- Threshold values for detecting spring/fall transitions ---
trend_threshold_conc = 0.5    # ice area fraction (0-1)
trend_threshold_thk  = 0.25   # effective ice thickness (m)

# --- Rolling-average window (days) applied before threshold detection ---
trend_rolling_window = 14

# --- DOY search windows, in SHIFTED space (0 = March 15) ---
# No wraparound syntax needed -- every range is a plain (start, end) with end > start.
# Use cyt.calendar_doy_to_shifted(calendar_doy) to translate a familiar calendar-DOY
# value into shifted space, e.g. cyt.calendar_doy_to_shifted(121) -> spring range start.
trend_spring_doy_range = (48, 109)    # roughly May - July (shifted)
trend_fall_doy_range   = (232, 323)   # roughly Nov - end of Jan (shifted; was the
                                       # problematic (305,365)/(350,30) calendar-year case)

# --- x-axis centering (shifted DOY to place at the centre of the heatmap x-axis) ---
# 185 (shifted) ~ Sept 16, near the annual Arctic sea-ice minimum.
plot_center_doy = 185

# --- Colormap limits ---
conc_vmin, conc_vmax = 0.0, 1.0
thk_vmin,  thk_vmax  = 0.0, 4.0

## Load the climate-year dataset

In [ ]:
import glob

if not os.path.exists(nc_path):
    # route_label above didn't match the file that was actually saved -- fall back
    # to auto-discovering whatever sea-ice climate-year dataset(s) exist in this folder.
    _candidates = sorted(glob.glob(os.path.join('.', 'seaice_climate_year_route*.nc')))
    if len(_candidates) == 1:
        nc_path = _candidates[0]
        print(f"NOTE: '{route_label}' did not match any file; "
              f"auto-selected the only dataset found: {nc_path}")
    elif len(_candidates) > 1:
        raise FileNotFoundError(
            f"route_label = '{route_label}' does not match any saved dataset.\n"
            f"Found multiple candidates -- set route_label to match one of these "
            f"filenames (the part after 'seaice_climate_year_route' and before '.nc'):\n"
            + '\n'.join(f'  {c}' for c in _candidates)
        )
    else:
        raise FileNotFoundError(
            f"No sea-ice climate-year dataset found in this folder matching "
            f"'{nc_path}', and no 'seaice_climate_year_route*.nc' files exist here "
            f"at all. Run the export cell in seaice_daily_heatmaps.ipynb first."
        )

ds = xr.open_dataset(nc_path)
print(ds)

_stat_key_map = {
    'median':          ('max_conc_med',  'max_thk_med'),
    '5th_percentile':  ('max_conc_5th',  'max_thk_5th'),
    '95th_percentile': ('max_conc_95th', 'max_thk_95th'),
}
_stat_label_map = {
    'median':          'Median',
    '5th_percentile':  '5th Percentile',
    '95th_percentile': '95th Percentile',
}
if trend_stat not in _stat_key_map:
    raise ValueError(
        f"trend_stat must be 'median', '5th_percentile', or '95th_percentile'; "
        f"got '{trend_stat}'"
    )
_conc_key, _thk_key = _stat_key_map[trend_stat]
_stat_label = _stat_label_map[trend_stat]

conc_data    = ds[_conc_key].values   # (n_seasons, 365)
thk_data     = ds[_thk_key].values    # (n_seasons, 365)
season_years = [int(y) for y in ds['season_year'].values]
n_seasons    = len(season_years)
year_arr     = np.array(season_years, dtype=float)

print(f'\nLoaded {_stat_label} concentration/thickness: '
      f'{n_seasons} season(s), {season_years[0]}\u2013{season_years[-1] + 1}')

## Spring / fall crossing detection, trend fit, scatter plot, and heatmap overlay

Runs for both Ice Concentration and Ice Thickness. Each heatmap row is a full ice
season (March 15 – March 14), so the fall- and spring-crossing markers are always
exactly one per row — no double or missing markers, unlike the calendar-year
heatmap overlay.

In [ ]:
_analysis_specs = [
    (conc_data, trend_threshold_conc, 'Ice Concentration', 'conc',
     'Blues_r', conc_vmin, conc_vmax, 'Ice area fraction (route-maximum)'),
    (thk_data,  trend_threshold_thk,  'Ice Thickness (m)',  'thk',
     'Blues_r', thk_vmin,  None,  'Effective ice thickness (m, route-maximum)'),
]

_start_doy = (plot_center_doy - 365 // 2) % 365

def _to_rolled(doy):
    return (doy - _start_doy) % 365

trend_results = {}

for data, threshold, field_label, field_slug, cmap_name, vmin, vmax, cbar_label in _analysis_specs:
    print(f'\n===== {field_label}  ({_stat_label})  threshold = {threshold} =====')

    smoothed = cyt.smooth_by_year(data, trend_rolling_window)

    spring_doys = np.full(n_seasons, np.nan)
    fall_doys   = np.full(n_seasons, np.nan)
    for yi in range(n_seasons):
        spring_doys[yi] = cyt.find_crossing_doy(
            smoothed[yi], threshold,
            trend_spring_doy_range[0], trend_spring_doy_range[1],
            find='last_above',   # ice concentration/thickness drops below threshold -> melt
        )
        fall_doys[yi] = cyt.find_crossing_doy(
            smoothed[yi], threshold,
            trend_fall_doy_range[0], trend_fall_doy_range[1],
            find='first_above',  # ice concentration/thickness rises above threshold -> freeze
        )

    print('  Spring (last day >= threshold before summer):')
    sp_slope, sp_int, sp_r2, sp_rmse, sp_yrs, sp_doys = cyt.compute_trend(spring_doys, year_arr)
    if not np.isnan(sp_slope):
        print(f'    slope = {sp_slope:+.2f} days/yr,  R\u00b2 = {sp_r2:.3f},  '
              f'RMSE = {sp_rmse:.1f} days,  n = {len(sp_yrs)}')

    print('  Fall (first day >= threshold after summer):')
    fa_slope, fa_int, fa_r2, fa_rmse, fa_yrs, fa_doys = cyt.compute_trend(fall_doys, year_arr)
    if not np.isnan(fa_slope):
        print(f'    slope = {fa_slope:+.2f} days/yr,  R\u00b2 = {fa_r2:.3f},  '
              f'RMSE = {fa_rmse:.1f} days,  n = {len(fa_yrs)}')

    trend_results[field_slug] = {
        'spring_doys': spring_doys, 'fall_doys': fall_doys,
        'sp_slope': sp_slope, 'sp_int': sp_int, 'sp_r2': sp_r2, 'sp_rmse': sp_rmse,
        'fa_slope': fa_slope, 'fa_int': fa_int, 'fa_r2': fa_r2, 'fa_rmse': fa_rmse,
    }

    # ---- Scatter plot ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'{field_label} (Climate Year)  \u2014  {_stat_label}  |  Route {route_label}\n'
        f'Threshold = {threshold}   Smoothing window = {trend_rolling_window} days',
        fontsize=12, fontweight='bold',
    )
    _season_specs = [
        ('Spring', 'Last day \u2265 threshold before summer', spring_doys,
         sp_slope, sp_int, sp_r2, sp_rmse),
        ('Fall',   'First day \u2265 threshold after summer', fall_doys,
         fa_slope, fa_int, fa_r2, fa_rmse),
    ]
    for ax, (season_name, season_desc, all_doys, slope, intercept, r2, rmse) in zip(axes, _season_specs):
        valid_mask = ~np.isnan(all_doys)
        ax.scatter(year_arr[valid_mask], all_doys[valid_mask],
                   color='steelblue', s=50, zorder=3, label='Observed')
        if not np.isnan(slope):
            fit_x = np.array([year_arr[valid_mask].min(), year_arr[valid_mask].max()])
            fit_y = slope * fit_x + intercept
            ax.plot(fit_x, fit_y, 'r-', linewidth=2, label=f'Trend: {slope:+.2f} d/yr')
            stats_text = f'R\u00b2 = {r2:.3f}\nRMSE = {rmse:.1f} d'
            ax.text(0.97, 0.05, stats_text, transform=ax.transAxes,
                   ha='right', va='bottom', fontsize=10,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))
        if valid_mask.any():
            _pad = 14
            _ylo = max(0,   float(np.nanmin(all_doys[valid_mask])) - _pad)
            _yhi = min(364, float(np.nanmax(all_doys[valid_mask])) + _pad)
        else:
            _ylo, _yhi = 0, 364
        _vis_ticks  = [d for d in cyt.SHIFTED_MONTH_DOY if _ylo <= d <= _yhi]
        _vis_labels = [cyt.SHIFTED_MONTH_NAMES[i] for i, d in enumerate(cyt.SHIFTED_MONTH_DOY) if _ylo <= d <= _yhi]
        ax.set_yticks(_vis_ticks)
        ax.set_yticklabels(_vis_labels, fontsize=9)
        ax.set_ylim(_ylo, _yhi)
        ax.yaxis.grid(True, alpha=0.3)
        ax.set_title(f'{season_name}: {season_desc}', fontsize=11)
        ax.set_xlabel('Season-start year', fontsize=10)
        ax.set_ylabel('Day of season (shifted, 0 = Mar 15)', fontsize=10)
        ax.set_xticks(year_arr.astype(int))
        ax.tick_params(axis='x', rotation=45)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_figs:
        os.makedirs(output_dir, exist_ok=True)
        _stat_slug = _stat_label.lower().replace(' ', '_').replace('th', '').replace('st', '')
        fname = os.path.join(output_dir, f'seaice_{field_slug}_trend_climate_year_route{route_label}_{_stat_slug}.png')
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'  Saved: {fname}')
    plt.show()

    # ---- Heatmap with trend overlay ----
    trend_overlay_arg = {}
    for season_key, all_doys, slope, intercept in [
        ('spring', spring_doys, sp_slope, sp_int),
        ('fall',   fall_doys,   fa_slope, fa_int),
    ]:
        valid = ~np.isnan(all_doys)
        trend_overlay_arg[season_key] = {
            'yi':        np.where(valid)[0].astype(float),
            'rolled_x':  np.array([_to_rolled(d) for d in all_doys[valid]]),
            'slope':     slope,
            'intercept': intercept,
        }

    fig, ax = cyt.plot_climate_year_heatmap_with_trend(
        data           = data,
        season_years   = season_years,
        stat_label     = _stat_label,
        field_label    = field_label,
        cmap_name      = cmap_name,
        vmin           = vmin,
        vmax           = vmax,
        cbar_label     = cbar_label,
        center_doy     = plot_center_doy,
        contour_levels = None,
        trend_overlay  = trend_overlay_arg,
        route_label    = route_label,
        save_figs      = save_figs,
        output_dir     = output_dir,
        field_slug      = f'seaice_{field_slug}',
    )
    plt.show()

print('\nClimate-year trend analysis complete.')